In [1]:
import pandas as pd
import ollama

MODEL = "llama3.2:3b"

response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "In one sentence, what is a manufacturing execution system?"}],
)

print(response["message"]["content"])

A Manufacturing Execution System (MES) is a software solution that tracks and manages the production process in real-time, providing detailed insights and data on manufacturing operations, quality control, inventory management, and supply chain logistics.


In [5]:
batches = pd.read_csv("../data/processed/batches.csv")
events = pd.read_csv("../data/processed/downtime_events.csv")
events = events.merge(batches[["Batch", "Operator"]], on="Batch", how="left")

total_time = batches["Duration"].sum()
total_lost = batches["Lost Time"].sum()
efficiency = batches["Min batch time"].sum() / total_time * 100

causes = events.groupby("Description")["Minutes"].sum().sort_values(ascending=False)
op_error_share = events.loc[events["Operator Error"] == "Yes", "Minutes"].sum() / total_lost * 100

facts = f"""Line: soda bottling line, {len(batches)} batches over {batches['Date'].nunique()} production days.
Total run time: {total_time/60:.1f} hours. Time lost to downtime: {total_lost/60:.1f} hours.
Line efficiency: {efficiency:.1f}%.
Operator errors caused {op_error_share:.1f}% of lost time; machines, materials and other causes caused {100-op_error_share:.1f}%.
Top downtime causes (minutes, share of lost time):
"""

cause_type = events.drop_duplicates("Description").set_index("Description")["Operator Error"]

for cause, minutes in causes.head(5).items():
    label = "operator error" if cause_type[cause] == "Yes" else "not operator error"
    facts += f"- {cause} ({label}): {minutes:.0f} min ({minutes/total_lost*100:.1f}%)\n"

print(facts)

Line: soda bottling line, 31 batches over 5 production days.
Total run time: 53.0 hours. Time lost to downtime: 18.8 hours.
Line efficiency: 64.5%.
Operator errors caused 51.6% of lost time; machines, materials and other causes caused 48.4%.
Top downtime causes (minutes, share of lost time):
- Machine failure (not operator error): 236 min (20.9%)
- Inventory shortage (not operator error): 205 min (18.1%)
- Machine adjustment (operator error): 197 min (17.4%)
- Batch change (operator error): 160 min (14.2%)
- Batch coding error (operator error): 115 min (10.2%)



In [8]:
facts += "\nOperator performance (per batch, fairer than totals since operators ran different numbers of batches):\n"

for op, group in batches.groupby("Operator"):
    op_events = events[(events["Operator"] == op) & (events["Operator Error"] == "Yes")]
    op_error_min = op_events["Minutes"].sum()
    top_error = op_events.groupby("Description")["Minutes"].sum().idxmax()
    lost_not_own = group["Lost Time"].sum() - op_error_min

    facts += (
        f"- {op}: ran {len(group)} batches. "
        f"Average lost time: {group['Lost Time'].mean():.1f} min per batch. "
        f"Biggest operator error: {top_error}, {op_events.groupby('Description')['Minutes'].sum().max():.0f} min in total across all their batches. "
        f"Lost to causes outside their control: {lost_not_own:.0f} min in total.\n"
    )

facts += "\nData note: only 31 batches, so operator differences are patterns worth investigating, not proof.\n"

print(facts)

Line: soda bottling line, 31 batches over 5 production days.
Total run time: 53.0 hours. Time lost to downtime: 18.8 hours.
Line efficiency: 64.5%.
Operator errors caused 51.6% of lost time; machines, materials and other causes caused 48.4%.
Top downtime causes (minutes, share of lost time):
- Machine failure (not operator error): 236 min (20.9%)
- Inventory shortage (not operator error): 205 min (18.1%)
- Machine adjustment (operator error): 197 min (17.4%)
- Batch change (operator error): 160 min (14.2%)
- Batch coding error (operator error): 115 min (10.2%)

Operator performance (per batch, fairer than totals since operators ran different numbers of batches):
- Charlie: 11 batches, 34.9 min lost per batch, 20.7 min of operator error per batch, biggest operator error: Machine adjustment (118 min), 156 min lost to causes outside their control
- Dee: 7 batches, 29.6 min lost per batch, 9.9 min of operator error per batch, biggest operator error: Batch change (20 min), 138 min lost to c

In [9]:
system_prompt = """You are a manufacturing performance analyst writing for a plant manager.
Rules:
- Use ONLY the facts provided. Never invent numbers, causes, or events.
- Copy numbers exactly as given.
- Each cause is labeled as operator error or not operator error. Keep these categories correct.
- Be fair to operators: mention time lost to causes outside their control, and treat operator differences as patterns to investigate, not proof.
- Write in plain English, short and practical.

Structure the report with these sections:
1. Summary (2-3 sentences)
2. Key findings: include one finding for EACH operator, naming their biggest operator error.
3. Recommendations: specific actions linked to the findings, including targeted training for specific operators.
4. Limitations of this data"""

response = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Write a performance report from these facts:\n\n{facts}"},
    ],
    options={"temperature": 0.2},
)

report = response["message"]["content"]
print(report)

**Performance Report: Soda Bottling Line**

**Summary**
Over 5 production days, the soda bottling line experienced a total run time of 53.0 hours, with 18.8 hours lost to downtime. Operator errors caused 51.6% of lost time, while machines, materials, and other causes accounted for 48.4%. The line efficiency was 64.5%.

**Key Findings**

* Charlie's biggest operator error was Machine adjustment, resulting in 118 minutes of lost time per batch. This accounts for 156 minutes of lost time across all their batches, with 138 minutes lost to causes outside their control.
* Dee's biggest operator error was Batch change, resulting in 20 minutes of lost time per batch. This accounts for 138 minutes of lost time across all their batches, with 138 minutes lost to causes outside their control.
* Dennis's biggest operator error was Machine adjustment, resulting in 50 minutes of lost time per batch. This accounts for 113 minutes of lost time across all their batches, with 113 minutes lost to causes o

In [11]:
best = batches.groupby("Operator")["Lost Time"].mean().idxmin()
line_avg_error = events.loc[events["Operator Error"] == "Yes", "Minutes"].sum() / len(batches)

findings, line_recs, op_recs = [], [], []

for op, group in batches.groupby("Operator"):
    op_events = events[(events["Operator"] == op) & (events["Operator Error"] == "Yes")]
    by_err = op_events.groupby("Description")["Minutes"].sum()
    outside = group["Lost Time"].sum() - by_err.sum()
    findings.append(
        f"{op}: {group['Lost Time'].mean():.1f} min lost per batch on average over {len(group)} batches. "
        f"Biggest own error: {by_err.idxmax()} ({by_err.max():.0f} min in total). "
        f"{outside:.0f} min lost to causes outside their control."
    )
    if by_err.sum() / len(group) > line_avg_error:
        op_recs.append(f"Targeted training for {op} on {by_err.idxmax().lower()}.")

actions = {
    "Machine failure": "Review preventive maintenance to reduce machine failures",
    "Inventory shortage": "Improve supply planning so materials are staged before batches start",
}
for cause in causes.index:
    if cause in actions and cause_type[cause] == "No":
        line_recs.append(f"{actions[cause]} ({causes[cause]:.0f} min lost).")

recs = line_recs + op_recs + [f"Use {best}'s methods as a model for other operators (lowest average lost time per batch)."]

print("KEY FINDINGS"); print("\n".join("- " + f for f in findings))
print("\nRECOMMENDATIONS"); print("\n".join("- " + r for r in recs))

KEY FINDINGS
- Charlie: 34.9 min lost per batch on average over 11 batches. Biggest own error: Machine adjustment (118 min in total). 156 min lost to causes outside their control.
- Dee: 29.6 min lost per batch on average over 7 batches. Biggest own error: Batch change (20 min in total). 138 min lost to causes outside their control.
- Dennis: 41.4 min lost per batch on average over 5 batches. Biggest own error: Machine adjustment (50 min in total). 113 min lost to causes outside their control.
- Mac: 41.5 min lost per batch on average over 8 batches. Biggest own error: Batch change (130 min in total). 140 min lost to causes outside their control.

RECOMMENDATIONS
- Review preventive maintenance to reduce machine failures (236 min lost).
- Improve supply planning so materials are staged before batches start (205 min lost).
- Targeted training for Charlie on machine adjustment.
- Targeted training for Mac on batch change.
- Use Dee's methods as a model for other operators (lowest average

In [12]:
import re

summary_facts = f"""Line efficiency: {efficiency:.1f}%.
Time lost to downtime: {total_lost/60:.1f} hours out of {total_time/60:.1f} hours.
Operator errors: {op_error_share:.1f}% of lost time. Other causes: {100-op_error_share:.1f}%.
Biggest cause: {causes.index[0]} ({causes.iloc[0]:.0f} min)."""

response = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Write a 2-3 sentence summary for a plant manager. Use only the facts given and copy numbers exactly. No headings, no bullet points."},
        {"role": "user", "content": summary_facts},
    ],
    options={"temperature": 0.2},
)
summary = response["message"]["content"].strip()

allowed = set(re.findall(r"\d+(?:\.\d+)?", summary_facts))
used = set(re.findall(r"\d+(?:\.\d+)?", summary))
unverified = used - allowed

print(summary)
print("\nNumbers not found in the facts:", unverified if unverified else "none ✅")

The plant's current efficiency is 64.5%, with a significant amount of lost time due to downtime, totaling 18.8 hours out of 53.0 hours. Operator errors account for 51.6% of the lost time, while machine failure is the largest contributor, causing 236 minutes of downtime.

Numbers not found in the facts: none ✅


In [13]:
import os
from datetime import date

report = f"""# Line Performance Report
Generated: {date.today()}

## Summary
{summary}

## Key Findings
{chr(10).join("- " + f for f in findings)}

## Recommendations
{chr(10).join("- " + r for r in recs)}

## Limitations
- Based on 31 batches over 5 production days, so operator differences are patterns worth investigating, not proof.
- 7 batches had downtime records but no production records, and were excluded.
- Summary written by a local AI model (Ollama, {MODEL}) and checked automatically for invented numbers. Findings and recommendations are calculated directly from the data.
"""

os.makedirs("../reports", exist_ok=True)
with open("../reports/performance_report.md", "w", encoding="utf-8") as f:
    f.write(report)

print(report)

# Line Performance Report
Generated: 2026-09-20

## Summary
The plant's current efficiency is 64.5%, with a significant amount of lost time due to downtime, totaling 18.8 hours out of 53.0 hours. Operator errors account for 51.6% of the lost time, while machine failure is the largest contributor, causing 236 minutes of downtime.

## Key Findings
- Charlie: 34.9 min lost per batch on average over 11 batches. Biggest own error: Machine adjustment (118 min in total). 156 min lost to causes outside their control.
- Dee: 29.6 min lost per batch on average over 7 batches. Biggest own error: Batch change (20 min in total). 138 min lost to causes outside their control.
- Dennis: 41.4 min lost per batch on average over 5 batches. Biggest own error: Machine adjustment (50 min in total). 113 min lost to causes outside their control.
- Mac: 41.5 min lost per batch on average over 8 batches. Biggest own error: Batch change (130 min in total). 140 min lost to causes outside their control.

## Recomm